# Data Mining Assignment: Apache Spark Data Lake Preprocessing
**Student Roll Number:** LCB2023039  
**Student Name:** Prashik Humane  
**Environment:** Google Colab + PySpark  
**Dataset:** sales.csv  

---

## Overview
This notebook establishes a simple **Data Lake architecture** featuring **Bronze** (raw storage) and **Silver** (cleaned/preprocessed storage) layers using Apache Spark (PySpark).

### Data Lake Architecture
```
data_lake/
├── bronze/
│   └── sales.csv             (Raw, unmodified dataset)
└── silver/
    └── preprocessed_sales/   (PySpark preprocessed output directory / CSV)
```

## Step 1: Install PySpark & Initialize SparkSession

In [ ]:
# Install PySpark
!pip install -q pyspark

In [ ]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Initialize SparkSession with legacy time parser policy for handling mixed date formats
spark = SparkSession.builder \
    .appName("DataLake_Preprocessing") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

print("SparkSession successfully started!")
print("Spark Version:", spark.version)

## Step 2: Set Up Data Lake Folders & Bronze Layer
We create the required `data_lake/bronze/` and `data_lake/silver/` directory structure and copy `sales.csv` into Bronze.

In [ ]:
# Create Data Lake folders
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver", exist_ok=True)

# Locate raw dataset and populate Bronze layer
source_file = "dataset/sales.csv" if os.path.exists("dataset/sales.csv") else "sales.csv"
target_bronze = "data_lake/bronze/sales.csv"

if os.path.exists(source_file):
    shutil.copy(source_file, target_bronze)
    print(f"Raw dataset copied to Bronze layer: {target_bronze}")
else:
    print(f"Ensure raw sales.csv is placed at {target_bronze}")

## Step 3: Read Raw Data from Bronze Layer & Inspect
Read `sales.csv` from Bronze into PySpark DataFrame, display schema, sample rows, and record count.

In [ ]:
# Read CSV into Spark DataFrame
df_bronze = spark.read.csv(target_bronze, header=True, inferSchema=True)

print("--- BRONZE LAYER SCHEMA ---")
df_bronze.printSchema()

print("\n--- BRONZE LAYER SAMPLE ROWS ---")
df_bronze.show(5, truncate=False)

bronze_count = df_bronze.count()
print(f"Total Bronze Record Count: {bronze_count}")

## Step 4: Identify Data Quality Discrepancies via Spark Operations
Using dynamic Spark queries (no hardcoding), we detect duplicates, missing values, text formatting issues, invalid numbers, and corrupted dates.

In [ ]:
print("================ DATA QUALITY DISCREPANCY ANALYSIS ================")

# 1. Exact Duplicates Count
distinct_count = df_bronze.dropDuplicates().count()
duplicate_rows_count = bronze_count - distinct_count
print(f"1. Exact Duplicate Rows: {duplicate_rows_count}")

# 2. Missing (Null) Values Count per Column
print("\n2. Null Value Counts per Column:")
null_df = df_bronze.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_bronze.columns])
null_df.show()

# 3. Text Format Inconsistencies (Capitalization & Extra Spaces)
print("3. Inconsistent Categories (Casing/Spacing sample):")
df_bronze.select("category").distinct().show()

# 4. Invalid Numerical Values
invalid_qty_cnt = df_bronze.filter(F.col("quantity") <= 0).count()
invalid_price_cnt = df_bronze.filter(F.col("unit_price") <= 0).count()
invalid_discount_cnt = df_bronze.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).count()

print(f"4. Invalid Numerical Values Detected:")
print(f"   - Quantity <= 0: {invalid_qty_cnt} rows")
print(f"   - Unit Price <= 0: {invalid_price_cnt} rows")
print(f"   - Discount % (< 0 or > 100): {invalid_discount_cnt} rows")

# 5. Inconsistent / Invalid Date Values
temp_date_parsed = F.coalesce(
    F.to_date(F.col("order_date"), "yyyy-MM-dd"),
    F.to_date(F.col("order_date"), "dd/MM/yyyy"),
    F.to_date(F.col("order_date"), "yyyy/MM/dd"),
    F.to_date(F.col("order_date"), "MM-dd-yyyy")
)
invalid_dates_cnt = df_bronze.filter(temp_date_parsed.isNull()).count()
print(f"\n5. Inconsistent / Invalid Date Values: {invalid_dates_cnt} rows")

## Step 5: Basic Preprocessing - Silver Layer
We perform the required cleaning transformations:
1. Remove exact duplicate rows (`dropDuplicates`).
2. Handle missing text values using `'Unknown'` imputation.
3. Trim whitespace and standardize text columns to Title Case (`initcap`).
4. Fix numerical anomalies (convert negative discount % to absolute, cap > 100% at 100%; filter `quantity > 0` and `unit_price > 0`).
5. Standardize order dates to `YYYY-MM-DD` and filter unparseable/invalid calendar dates.

In [ ]:
# 1. Deduplication
df_silver = df_bronze.dropDuplicates()

# 2. Missing Value Imputation
df_silver = df_silver.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown"
})

# 3. Text Field Standardization (Trim & Title Case)
text_columns = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]
for c in text_columns:
    df_silver = df_silver.withColumn(c, F.initcap(F.trim(F.col(c))))

# 4. Numerical Validation & Fixes
df_silver = df_silver.withColumn(
    "discount_percent",
    F.when(F.col("discount_percent") < 0, F.abs(F.col("discount_percent")))
     .when(F.col("discount_percent") > 100, 100)
     .otherwise(F.col("discount_percent"))
)
df_silver = df_silver.filter((F.col("quantity") > 0) & (F.col("unit_price") > 0))

# 5. Date Parsing & Standardization (YYYY-MM-DD)
parsed_date_col = F.coalesce(
    F.to_date(F.col("order_date"), "yyyy-MM-dd"),
    F.to_date(F.col("order_date"), "dd/MM/yyyy"),
    F.to_date(F.col("order_date"), "yyyy/MM/dd"),
    F.to_date(F.col("order_date"), "MM-dd-yyyy")
)
df_silver = df_silver.withColumn("order_date_parsed", parsed_date_col) \
                     .filter(F.col("order_date_parsed").isNotNull()) \
                     .withColumn("order_date", F.date_format(F.col("order_date_parsed"), "yyyy-MM-dd")) \
                     .drop("order_date_parsed")

print("Silver Layer Preprocessing Completed Successfully!")

## Step 6: Save Processed Data to Silver Layer
We save the clean DataFrame into `data_lake/silver/preprocessed_sales` and export a copy as `silver/sales_silver.csv`.

In [ ]:
# Save as partitioned PySpark output directory
silver_spark_path = "data_lake/silver/preprocessed_sales"
df_silver.coalesce(1).write.csv(silver_spark_path, header=True, mode="overwrite")
print(f"Saved Silver output directory: {silver_spark_path}")

# Save single CSV output file for submission
os.makedirs("silver", exist_ok=True)
for f in os.listdir(silver_spark_path):
    if f.startswith("part-") and f.endswith(".csv"):
        shutil.copy(os.path.join(silver_spark_path, f), "silver/sales_silver.csv")
        print("Copied Silver single CSV to: silver/sales_silver.csv")
        break

## Step 7: Verification & Results Summary
Display record counts before vs after preprocessing, state operations performed, and read back the Silver layer file.

In [ ]:
silver_count = df_silver.count()

print("==================== PREPROCESSING SUMMARY ====================")
print(f"Bronze Layer Row Count (Before Preprocessing) : {bronze_count}")
print(f"Silver Layer Row Count (After Preprocessing)  : {silver_count}")
print(f"Total Records Removed / Filtered             : {bronze_count - silver_count}")
print("===============================================================")

print("\n--- SUMMARY OF PREPROCESSING OPERATIONS PERFORMED ---")
print("1. Removed exact duplicate rows.")
print("2. Imputed missing customer_name, payment_method, and city with 'Unknown'.")
print("3. Trimmed leading/trailing spaces and standardized casing (Title Case) across text columns.")
print("4. Fixed discount_percent anomalies (negative values converted to positive, capped >100% at 100%).")
print("5. Filtered out corrupted records with quantity <= 0 or unit_price <= 0.")
print("6. Standardized valid date strings to YYYY-MM-DD and removed invalid calendar dates.")

print("\n--- READ BACK SILVER DATASET WITH SPARK ---")
df_silver_read = spark.read.csv("silver/sales_silver.csv", header=True, inferSchema=True)
df_silver_read.printSchema()
df_silver_read.show(10, truncate=False)